# Creating Handcrafted Features
We will add some handcrafted features to improve the performance of our logistic regression models. We will aim to add 3 general features along with 2 features that are more sentiment focused.

In [1]:
%pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import os
import kagglehub
from kagglehub import KaggleDatasetAdapter

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from collections import Counter

import string

c:\Users\stian\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Datasets Without Handcrafted Features

In [3]:
TRAIN_FILE_PATH = os.path.join('no_handcrafted', 'mental_health_text_train.csv')
TEST_FILE_PATH = os.path.join('no_handcrafted', 'mental_health_text_test.csv')


df_train = pd.read_csv(TRAIN_FILE_PATH, index_col=0)
df_test = pd.read_csv(TEST_FILE_PATH, index_col=0)

## Load Depression and Happiness Datasets

In [4]:
depression_path = kagglehub.dataset_download('diegosilvadefrana/depression-dataset')

df_depression = pd.read_csv(f'{depression_path}/dataset.csv', encoding='utf8')

df_depression.head()

,title,content,score
0,"Regular check-in post, with information about ...",Welcome to /r/depression's check-in post - a p...,115
1,Our most-broken and least-understood rules is ...,We understand that most people who reply immed...,2365
2,Going back to college at 33 after 3 times of d...,"I've always wanted to go back to school, the c...",84
3,Crying alone all the time,All I wan is to be loved. I just want someone...,108
4,I genuinely don’t think I’ll ever be able to h...,"I don’t even drive, didn’t finish my high scho...",60


In [5]:
happy_path = kagglehub.dataset_download("ritresearch/happydb")

df_happy = pd.read_csv(f'{happy_path}/cleaned_hm.csv', encoding='utf8')

df_happy.head()

,hmid,wid,reflection_period,original_hm,cleaned_hm,modified,num_sentence,ground_truth_category,predicted_category
0,27673,2053,24h,I went on a successful date with someone I fel...,I went on a successful date with someone I fel...,True,1,NaN,affection
1,27674,2,24h,I was happy when my son got 90% marks in his e...,I was happy when my son got 90% marks in his e...,True,1,NaN,affection
2,27675,1936,24h,I went to the gym this morning and did yoga.,I went to the gym this morning and did yoga.,True,1,NaN,exercise
3,27676,206,24h,We had a serious talk with some friends of our...,We had a serious talk with some friends of our...,True,2,bonding,bonding
4,27677,6227,24h,I went with grandchildren to butterfly display...,I went with grandchildren to butterfly display...,True,1,NaN,affection


In [6]:
df_depression.isnull().sum()

df_depression = df_depression.dropna()

In [7]:
top_depression = df_depression.sort_values('score', ascending=False).head(10)
for i in range(10):
  print(f'#{i+1} ------------------------------------------------------')
  print(top_depression.iloc[i]['content'])

#1 ------------------------------------------------------
Literally what the title says, I’m in shock rn idek what to say really. We’ve been on 4 dates and instantly clicked and had so much in common and constantly talked and he was just awesome, had sex on our third date and it was amazing as well. I really thought I had found a good one. And then right before our 5th date he went radio silent and promptly stood me up at the restaurant. Or so I thought. I thought “oh well he probably got something better to do thank god it’s only been like 3 weeks and I didn’t get too invested” but I also really liked him and was hella mad. 4 whole days of me delving into every insecurity I ever had trying to find a reason he would ghost me like this and embarrass me by standing me up.

And then I met my friend (his coworker) who introduced him to me. I didn’t want to seem unpleasant or anything so I just told him to tell the guy I’m not mad that he stood me up, to then be met with the saddest look.



Find the most common words

In [8]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

def get_frequency(df, content_name):
  '''
  Counts frequency of words and returns counter for all words
  '''

  text = ' '.join(df[content_name].dropna())
  tokens = word_tokenize(text.lower())
    
  ## Remove common stopwords
  stop_words = set(stopwords.words('english'))

  cleaned_tokens = []
  for token in tokens:
    if token not in stop_words and token.isalpha():
      cleaned_tokens.append(token)
  
  return Counter(cleaned_tokens)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\stian\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\stian\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\stian\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [9]:
dep_freq = get_frequency(df_depression, 'content')

happy_freq = get_frequency(df_happy, 'cleaned_hm')

In [10]:
dep_words = []
for word in dep_freq.most_common(25):
  dep_words.append(word[0])

happy_words = []
for word in happy_freq.most_common(25):
  happy_words.append(word[0])


In [11]:
def count_from_list(tokens, word_list):
  count = 0
  
  for token in tokens:
    if token in word_list:
      count += 1

  return count

def count_punctuation(tokens):
  count = 0
  for token in tokens:
    if token in string.punctuation:
      count += 1
  return count


In [12]:
text_len = []
token_count = []
punct_count = []
dep_count = []
happy_count = []

for row in df_train.values:
  text = row[0]
  tokens = word_tokenize(text)
  
  text_len.append(len(text))

  token_count.append(len(tokens))

  punct_count.append(count_punctuation(tokens))

  dep_count.append(count_from_list(tokens, dep_words))
  happy_count.append(count_from_list(tokens, happy_words))

df_train['text_length'] = text_len
df_train['token_count'] = token_count
df_train['punctuation_count'] = punct_count
df_train['depression_word_count'] = dep_count
df_train['happy_word_count'] = happy_count

In [13]:
text_len_test = []
token_count_test = []
punct_count_test = []
dep_count_test = []
happy_count_test = []

for i, row in enumerate(df_test.values):
  text = row[0]
  tokens = word_tokenize(text)

  text_len_test.append(len(text))

  token_count_test.append(len(tokens))

  punct_count_test.append(count_punctuation(tokens))
  
  dep_count_test.append(count_from_list(tokens, dep_words))
  happy_count_test.append(count_from_list(tokens, happy_words))

df_test['text_length'] = text_len_test
df_test['token_count'] = token_count_test
df_test['punctuation_count'] = punct_count_test
df_test['depression_word_count'] = dep_count_test
df_test['happy_word_count'] = happy_count_test

In [14]:
df_train.head()

,text,status,text_length,token_count,punctuation_count,depression_word_count,happy_word_count
3335,I have typed a message but I haven't yet sent ...,Normal,56,15,1,0,0
43240,gnite twitter world long day tomorrow night cl...,Normal,58,10,0,1,2
22622,"I am only 16 and I fucked up, I have a 1.2 GPA...",Suicidal,754,169,13,16,3
5526,"It's sad, when the electricity goes out while ...",Normal,97,20,1,0,0
30976,i can't believe how hot it is.,Normal,30,9,1,0,0


In [15]:
df_test.head()

,text,status,text_length,token_count,punctuation_count,depression_word_count,happy_word_count
29619,I make a good living and only want to get on w...,Normal,320,70,4,7,3
15959,I am a teenager. I want to cut myself. You can...,Suicidal,359,88,13,3,2
7391,Ã¢â¬ÅVisualizing your dreams down to the det...,Normal,130,21,3,1,1
5026,Hopefully SM will be more open and see the pot...,Normal,217,43,4,2,3
25225,I am not super suicidal right now but I still ...,Suicidal,421,96,5,6,0


In [17]:
dir_path = './handcrafted/'
df_train.to_csv(f'{dir_path}/mental_health_text_train_features.csv')
df_test.to_csv(f'{dir_path}/mental_health_text_test_features.csv')